<a href="https://colab.research.google.com/github/shanchepngen-ux/ML-For-Engineering/blob/main/Week3_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Problem and Intended AI/ML Outcome

## Project Problem

Artisanal and small-scale mining activities are often carried out with limited geotechnical planning, engineering monitoring and safety infrastructure. As a result, unstable slopes and mine-pit walls can fail, particularly during or after periods of heavy rainfall. Such failures can cause injuries, fatalities and significant damage to mining communities.

This project aims to investigate how Machine Learning can be used to identify areas that are susceptible to slope instability and landslides in and around artisanal mining areas.

## Intended AI/ML Outcome

The intended AI/ML system will use environmental and geospatial information to classify areas according to their landslide or slope-instability susceptibility.

The longer-term objective is to develop a system that could classify active artisanal mining areas into risk categories such as:

* Low Risk
* Medium Risk
* High Risk

and eventually estimate the probability of a mine-pit or surrounding slope failure following environmental triggers such as heavy rainfall.

For this assignment, a publicly available landslide-susceptibility dataset from Hugging face will be used as a relevant proxy dataset because a sufficiently large publicly available dataset specifically containing labelled artisanal mine-pit collapses was not identified. The approach developed using this dataset could subsequently be adapted to artisanal mining by incorporating mine locations, pit geometry, rainfall, soil properties, geological information and satellite-derived ground deformation.


# Data Required

To develop the proposed artisanal mining hazard prediction system, several categories of data would be required.

### 1. Topographic Data

* Digital Elevation Model (DEM)
* Elevation
* Slope angle
* Aspect
* Terrain curvature
* Drainage characteristics

These variables can help describe the physical characteristics and stability of a mining slope.

### 2. Weather and Environmental Data

* Daily rainfall
* Cumulative rainfall over 3, 7 and 14 days
* Soil moisture
* Temperature
* Surface water conditions

Rainfall and soil moisture are particularly important because water infiltration can increase pore-water pressure and reduce slope stability.

### 3. Geological and Soil Data

* Soil type
* Clay, sand and silt content
* Geological formation
* Rock type
* Geological structures such as faults where available

These variables can help describe the strength and behaviour of the material forming the slope.

### 4. Remote Sensing Data

* Sentinel-1 SAR data
* Sentinel-2 multispectral imagery
* Surface deformation
* Land-cover change
* Mining disturbance

Satellite data could be used to monitor changes in the ground surface and identify areas of increasing instability.

### 5. Mining-Specific Data

* Location of mining pits
* Pit depth
* Pit width
* Pit-wall slope angle
* Excavation method
* Presence of support structures
* Water accumulation in pits
* Previous mine failures

### 6. Historical Hazard Data

* Historical landslide locations
* Historical mine-collapse locations
* Date of each failure
* Location coordinates
* Environmental conditions before the failure

These historical events would provide the target labels required for supervised machine learning.

For this assignment, a publicly available landslide-susceptibility dataset from Hugging Face will be used as a proxy because it contains relevant environmental and geospatial information and a hazard-related target variable.


In [9]:
from datasets import load_dataset

ds = load_dataset("ZefanCai/Open-Jev", "release-v2-redistributable")

README.md:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

data/release-v2-redistributable/train-00(…): reconstructing file:   0%|          |  0.00B / 5.03MB            

data/release-v2-redistributable/train-00(…): downloading bytes:           |  0.00B            

data/release-v2-redistributable/calibrat(…): reconstructing file:   0%|          |  0.00B /  382kB            

data/release-v2-redistributable/calibrat(…): downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/79116 [00:00<?, ? examples/s]

Generating calibration split:   0%|          | 0/4672 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3723 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10356 [00:00<?, ? examples/s]

Generating ood split:   0%|          | 0/15701 [00:00<?, ? examples/s]

In [12]:
import pandas as pd

# Convert the loaded Hugging Face dataset 'ds' (from the previous cell) to a pandas DataFrame
# Depending on the split names inside 'ds', we can convert the 'train' split
if 'train' in ds:
    df = pd.DataFrame(ds['train'])
    print("Successfully loaded train split into a DataFrame!")
    print(df.head())
else:
    # If 'train' split doesn't exist, print available splits
    print("Available splits in dataset:", list(ds.keys()))

Successfully loaded train split into a DataFrame!
                                                id                  group_id  \
0                customer-control-v1:42:1:category  customer-control-v1:42:1   
1            customer-control-v1:42:1:bug_severity  customer-control-v1:42:1   
2  customer-control-v1:42:1:has_reproducible_steps  customer-control-v1:42:1   
3        customer-control-v1:42:1:refund_requested  customer-control-v1:42:1   
4             customer-control-v1:42:1:frustration  customer-control-v1:42:1   

   split               source    kind  \
0  train  customer-control-v1  choice   
1  train  customer-control-v1   score   
2  train  customer-control-v1    noul   
3  train  customer-control-v1    noul   
4  train  customer-control-v1   score   

                                            question  \
0  Determine the broad category of this support t...   
1  How severe is the reported issue? Use the stat...   
2  The user describes specific steps to reproduce...  

In [13]:
df.head()

,id,group_id,split,source,kind,question,options,target,state_json,metadata_json,record_json,original_line_number
0,customer-control-v1:42:1:category,customer-control-v1:42:1,train,customer-control-v1,choice,Determine the broad category of this support t...,"[account: Login, permissions, profile, securit...","[1.0, 0.0, 0.0, 0.0]","""Conversation with account holder Harbor-65673...","{""case_name"": ""customer_support_fanout_control...","{""group_id"": ""customer-control-v1:42:1"", ""id"":...",1
1,customer-control-v1:42:1:bug_severity,customer-control-v1:42:1,train,customer-control-v1,score,How severe is the reported issue? Use the stat...,"[Cosmetic; no impact to functionality, Broken ...","[1.0, 0.0, 0.0]","""Conversation with account holder Harbor-65673...","{""case_name"": ""customer_support_fanout_control...","{""group_id"": ""customer-control-v1:42:1"", ""id"":...",2
2,customer-control-v1:42:1:has_reproducible_steps,customer-control-v1:42:1,train,customer-control-v1,noul,The user describes specific steps to reproduce...,"[no, yes]","[1.0, 0.0]","""Conversation with account holder Harbor-65673...","{""case_name"": ""customer_support_fanout_control...","{""group_id"": ""customer-control-v1:42:1"", ""id"":...",3
3,customer-control-v1:42:1:refund_requested,customer-control-v1:42:1,train,customer-control-v1,noul,The user is explicitly asking for a refund or ...,"[no, yes]","[1.0, 0.0]","""Conversation with account holder Harbor-65673...","{""case_name"": ""customer_support_fanout_control...","{""group_id"": ""customer-control-v1:42:1"", ""id"":...",4
4,customer-control-v1:42:1:frustration,customer-control-v1:42:1,train,customer-control-v1,score,How frustrated the user appears. Use the curre...,"[Calm, matter-of-fact, Frustrated but civil, V...","[0.0, 1.0, 0.0]","""Conversation with account holder Harbor-65673...","{""case_name"": ""customer_support_fanout_control...","{""group_id"": ""customer-control-v1:42:1"", ""id"":...",5


In [14]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 79116
Number of columns: 12


In [15]:
df.describe()


,original_line_number
count,79116.000000
mean,41031.592674
std,23184.233781
min,1.000000
25%,21479.750000
50%,41258.500000
75%,61037.250000
max,80816.000000


# Potential Inputs (Features) and Output (Target)

## Potential Input Features

The potential input features for the model are the environmental and geospatial variables contained in the selected dataset. These may include variables related to:

* Topography and elevation
* Slope
* Rainfall or precipitation
* Soil and environmental conditions
* Satellite-derived characteristics
* Other geographic variables provided in the dataset

These variables represent factors that may influence slope instability and landslide occurrence.

## Target Variable

The target variable will represent the observed landslide susceptibility or landslide occurrence provided by the dataset.

The model will learn the relationship between the environmental/geospatial input variables and the observed hazard label.

In the eventual artisanal-mining application, the target would ideally represent whether a mining pit or slope experienced structural failure within a specified period.


# Proposed Machine Learning Workflow

```text
                  PROJECT DATA
                       │
                       ▼
        ┌─────────────────────────────┐
        │ Environmental & Geospatial  │
        │          Data               │
        │                             │
        │ • Rainfall                  │
        │ • Elevation                 │
        │ • Slope                     │
        │ • Soil characteristics      │
        │ • Satellite information     │
        └──────────────┬──────────────┘
                       │
                       ▼
                 DATA CLEANING
                       │
                       ▼
              DATA PREPROCESSING
                       │
             ┌─────────┴─────────┐
             │                   │
             ▼                   ▼
       Feature selection    Missing values
             │              /normalisation
             └─────────┬─────────┘
                       │
                       ▼
                TRAINING DATA
                       │
                       ▼
                ML ALGORITHM
             ┌─────────┴─────────┐
             │                   │
             ▼                   ▼
        Random Forest         XGBoost
             │                   │
             └─────────┬─────────┘
                       │
                       ▼
                  PREDICTION
                       │
                       ▼
             LANDSLIDE / HAZARD
                 SUSCEPTIBILITY
                       │
                       ▼
                  EVALUATION
                       │
             ┌─────────┼─────────┐
             ▼         ▼         ▼
          Accuracy   Recall    F1-score
```

## Workflow Explanation

First, the available environmental and geospatial data will be collected and imported into the Python environment. The data will then be inspected and cleaned to identify missing values, inconsistent observations and unsuitable variables.

Relevant environmental and geospatial variables will be selected as input features. The dataset will then be divided into training and testing data.

Machine Learning algorithms such as Random Forest and XGBoost can be trained using the selected features. These algorithms will learn relationships between environmental conditions and the observed landslide hazard.

The trained model will then generate predictions for previously unseen data. Model performance can be evaluated using appropriate classification metrics such as accuracy, precision, recall and F1-score.

In the future, the same workflow could be extended to artisanal mining by adding mining-specific information such as pit geometry, excavation depth, historical mine-collapse events, rainfall accumulation and satellite-derived ground deformation. The final system could then produce risk categories for individual mining areas and support early warning and safety decisions.
